# OpenDDE Structure Prediction

![OpenDDE Structure Prediction](https://proto-bio.github.io/proto-assets/images/tool/opendde/hero.png)

This notebook demonstrates all-atom structure prediction using OpenDDE, an open-source co-folding model developed by Aureka AI Research. OpenDDE jointly predicts the three-dimensional structures of proteins, nucleic acids (DNA and RNA), small-molecule ligands, and their complexes using a diffusion-based generative approach with configurable sampling and recycling steps. We demonstrate `run_opendde` by folding the Trp-cage mini-protein and human insulin, covering input construction, model configuration, result inspection, visualization, and export.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("opendde")
display_overview("opendde")
display_docs_section("opendde", "Background")

## Available tools

In [ ]:
display_available_tools("opendde")

### `run_opendde`

OpenDDE predicts 3D structures of proteins, DNA, RNA, ligands, and their complexes using a diffusion-based generative model. It supports both single-sequence mode and MSA-assisted prediction via MMseqs2 homology search. The `num_cycles` parameter controls iterative structural refinement, while `num_steps` governs the granularity of the denoising process. When `num_samples` is set above 1, OpenDDE generates multiple independent structure samples and returns the best by ranking score, which is useful for exploring conformational diversity. Ligands are provided as SMILES strings or CCD codes and are automatically converted to the appropriate internal representation.

In [ ]:
from pathlib import Path

from proto_tools import (
    Chain,
    Complex,
    OpenDDEConfig,
    OpenDDEInput,
    run_opendde,
)

In [ ]:
# Display input docs
display_api_reference("opendde", "input", "run_opendde")

# Trp-cage TC5b — a 20-residue mini-protein that adopts a compact fold
trpcage_sequence = "NLYIQWLKDGGPSSGRPPPS"

# Create a single-protein complex
complex = Complex(chains=[Chain(sequence=trpcage_sequence, entity_type="protein")])

# Create input
inputs = OpenDDEInput(complexes=[complex])

In [ ]:
# Display config docs
display_api_reference("opendde", "config", "run_opendde")

# Configure OpenDDE with reduced settings for a fast demonstration run
config = OpenDDEConfig(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
    use_msa=False,  # single-sequence mode for a fast first prediction
    num_samples=1,
)

In [ ]:
# Run structure prediction
result = run_opendde(inputs, config)

In [ ]:
# Display output docs
display_api_reference("opendde", "output", "run_opendde")

trpcage_structure = result.structures[0]

# Print confidence metrics
print(f"  Number of chains:  {len(complex.chains)}")
print(f"  Protein length:    {len(trpcage_sequence)} residues")
print(f"  Average pLDDT:     {trpcage_structure.metrics.avg_plddt:.1f}")
print(f"  pTM score:         {trpcage_structure.metrics.ptm:.3f}")
print(f"  Ranking score:     {trpcage_structure.metrics.ranking_score:.3f}")

#### Visualize the predicted structure

The interactive viewer renders the predicted mini-protein colored by pLDDT confidence, allowing you to inspect the fold geometry and per-residue confidence.

> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [ ]:
trpcage_structure.visualize(style="cartoon", color_by="bfactor")

#### Predict a multi-chain complex

OpenDDE can fold multi-chain complexes and report an interface confidence (`iptm`) between chains. Here we fold human insulin, a two-chain complex whose A and B chains associate through disulfide bonds, and tune the sampling configuration: `num_samples=2` keeps the best of two diffusion samples (by ranking score), while `num_steps=100` trades a little accuracy for speed.

In [ ]:
# Human insulin — two-chain protein complex (A chain + B chain)
insulin_a_chain = "GIVEQCCTSICSLYQLENYCN"
insulin_b_chain = "FVNQHLCGSHLVEALYLVCGERGFFYTPKT"

insulin_complex = Complex(
    chains=[
        Chain(sequence=insulin_a_chain, entity_type="protein"),
        Chain(sequence=insulin_b_chain, entity_type="protein"),
    ]
)
insulin_inputs = OpenDDEInput(complexes=[insulin_complex])

insulin_config = OpenDDEConfig(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
    model_name="opendde_v1",
    num_samples=2,  # keep the best of 2 diffusion samples by ranking score
    num_steps=100,  # fewer denoising steps for a faster demo
    use_msa=False,
)

insulin_result = run_opendde(insulin_inputs, insulin_config)
insulin_structure = insulin_result.structures[0]
print(f"  Chains:         {len(insulin_complex.chains)}")
print(f"  Average pLDDT:  {insulin_structure.metrics.avg_plddt:.1f}")
print(f"  pTM score:      {insulin_structure.metrics.ptm:.3f}")
print(f"  ipTM score:     {insulin_structure.metrics.iptm:.3f}")  # interface confidence between chains

## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD. The B-factor column contains pLDDT confidence scores for per-residue visualization.

In [ ]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to mmCIF (B-factor column carries per-residue pLDDT)
insulin_result.export(name="insulin_complex", export_path=output_dir, file_format="cif")

# Export results to PDB for tools like PyMOL / ChimeraX
insulin_result.export(name="insulin_complex", export_path=output_dir, file_format="pdb")
print(f"Structure exported to {output_dir / 'insulin_complex.cif'}")